# LLM Fundamentals

---

## 01 — LLM Mental Model

Before writing any code, we need to understand the high-level path that text follows inside a Large Language Model (LLM).

A user sees:

```text
"I love machine learning."
```

But the model does not process this sentence as human-readable text.

Instead, the text is transformed into numerical representations before the Transformer can process it.

The overall pipeline is:

```text
User Text
   ↓
Tokenizer
   ↓
Tokens
   ↓
Token IDs
   ↓
Embeddings
   ↓
Position Information
   ↓
Decoder-Only Transformer
   ↓
Hidden States
   ↓
Language Modeling Head
   ↓
Logits
   ↓
Next-Token Prediction
```

For the Local AI Assistant, this pipeline is the foundation of the entire inference process.

The first part of this pipeline is:

```text
Text
 ↓
Tokenization
 ↓
Token IDs
 ↓
Embeddings
 ↓
Transformer Input
```

The important mental model is:

> **Language enters the model as numbers, not as raw human-readable text.**

This means that an LLM must first transform language into representations that neural network operations can process.

---

## 02 — What Exactly Is a Token?

A **token** is a unit produced by the tokenizer.

A token is not necessarily:

```text
1 token = 1 word
```

and it is not necessarily:

```text
1 token = 1 character
```

A token can represent:

- A whole word
- A part of a word
- Punctuation
- Whitespace-related pieces
- A number
- A symbol
- A special token

Therefore:

```text
1 token ≠ 1 word
1 token ≠ 1 character
```

The exact tokenization behavior depends on the tokenizer associated with the model.

For example, a word such as:

```text
unbelievable
```

may be represented using multiple subword pieces:

```text
un | believ | able
```

This is only an illustrative example.

The exact token boundaries depend on the tokenizer being used.

The important concept is:

```text
Text
 ↓
Tokenizer
 ↓
Token sequence
```

A token can therefore represent a reusable piece of language rather than necessarily representing an entire word.

---

## 03 — Why Tokenization Exists

Neural networks operate on numerical tensors.

Raw text such as:

```text
"Machine Learning"
```

cannot simply be passed directly into the model as a string.

The text must first be converted into a sequence of tokens:

```text
Raw Text
   ↓
Tokenization
   ↓
Token Sequence
```

Then the token sequence is converted into integer IDs:

```text
Token Sequence
   ↓
Token IDs
```

The model can then operate on these numerical representations.

The complete simplified flow is:

```text
Text
 ↓
Tokens
 ↓
Token IDs
 ↓
Embeddings
 ↓
Transformer
```

Tokenization therefore acts as the bridge between human language and the numerical input representation used by the model.

---

## 04 — Example: Tokenization

Consider:

```text
"Machine learning is useful."
```

A tokenizer could conceptually produce something like:

```text
["Machine", " learning", " is", " useful", "."]
```

The exact result is model-dependent.

One important observation is that spaces may be represented as part of token boundaries.

For example:

```text
"Machine"
" learning"
```

may be treated as different token pieces.

Do not memorize a specific tokenization output as universal.

The correct mental model is:

```text
Same Text
   ↓
Different Tokenizer
   ↓
Possibly Different Tokens
```

This means that tokenization is a property of the tokenizer/model pair, not a universal property of language.

---

## 05 — Why Tokens Are Useful

Subword tokenization allows the vocabulary to reuse pieces instead of storing every possible word independently.

For example, a tokenizer does not need a separate vocabulary entry for every imaginable variation of a word.

Instead, it can reuse smaller pieces.

This helps the model handle:

- Rare words
- New words
- Morphological variations
- Technical terminology
- Source code
- Mixed-language text

Conceptually:

```text
Many Possible Words
        ↓
Reusable Token Pieces
        ↓
Efficient Vocabulary
```

This is one reason tokenization is such an important part of modern LLM systems.

---

## 06 — Tokenization and Vocabulary

The tokenizer works together with a **vocabulary**.

Conceptually:

```text
Vocabulary
────────────────
token A
token B
token C
token D
...
token N
```

Each vocabulary item is associated with an integer identifier.

For example, conceptually:

```text
"machine"   → 1234
"learning"  → 5678
"."         → 42
```

These numbers are only illustrative.

The actual IDs depend entirely on the tokenizer and model.

The relationship is:

```text
Token
   ↓
Vocabulary Lookup
   ↓
Integer ID
```

The vocabulary therefore defines the set of token units that the tokenizer can map to IDs.

---

## 07 — Token IDs

After tokenization, the token sequence is mapped to integer IDs.

Conceptually:

```text
Tokens
   ↓
Integer IDs
```

For example:

```text
["I", "love", "AI"]
```

could conceptually become:

```text
[101, 532, 7281]
```

Again, these values are illustrative.

The important relationship is:

```text
Token
   ↓
Token ID
```

The token ID is an integer identifier assigned to a token in the tokenizer vocabulary.

These IDs are what the model uses as indices into its embedding representation.

---

## 08 — Why Integer IDs?

The model needs a compact numerical representation for each token.

Token IDs provide an efficient way to identify vocabulary entries.

Conceptually:

```text
Token ID
   ↓
Embedding Matrix Lookup
   ↓
Embedding Vector
```

For example:

```text
"machine"
    ↓
Token ID
    ↓
Embedding Lookup
    ↓
Vector
```

An important distinction is:

> **A token ID is not the semantic representation of a token.**

The ID is simply an index.

The learned semantic representation comes from the embedding vectors and later contextual representations produced by the Transformer.

---

## 09 — Special Tokens

Modern tokenizers may include special tokens that are used to control or represent structural information.

Examples can include:

```text
BOS
EOS
PAD
EOT
Other model-specific control tokens
```

The exact set and meaning depend on the tokenizer/model family.

Important:

```text
Not every model uses the same special tokens.
```

Special tokens may be used to indicate things such as:

- Start of a sequence
- End of a sequence
- Padding
- End of a turn
- Conversation structure

We should therefore inspect the actual tokenizer rather than assuming a universal set of special tokens.

---

## 10 — Embeddings

An **embedding** is a numerical vector representing a token.

Conceptually:

```text
"machine"
    ↓
[0.12, -0.44, 0.91, ..., 0.07]
```

The real vectors contain many dimensions.

We do not manually choose these numbers.

The pretrained model contains learned parameters that map token IDs to embedding vectors.

The simplified flow is:

```text
Token ID
   ↓
Embedding Lookup
   ↓
Embedding Vector
```

This is the stage where token identifiers are transformed into continuous numerical representations that can participate in neural network computation.

---

## 11 — Embedding Matrix

The model contains an embedding table, which can be viewed conceptually as a matrix.

Suppose:

```text
V = Vocabulary Size
D = Embedding Dimension
```

Then the embedding matrix has a conceptual shape:

```text
[V, D]
```

A simplified representation is:

```text
                 Embedding Dimension
                         ↓

Token 1      [ ... ... ... ... ]
Token 2      [ ... ... ... ... ]
Token 3      [ ... ... ... ... ]
Token 4      [ ... ... ... ... ]
   ...
Token V      [ ... ... ... ... ]
```

Each row corresponds to a vocabulary item.

A token ID selects one row from this matrix.

Therefore:

```text
Token ID
   ↓
Row Lookup
   ↓
Embedding Vector
```

This is an important bridge between tokenization and neural network computation.

---

## 12 — Token vs Token ID vs Embedding vs Hidden State

These four concepts must not be confused.

### Token

A token is the tokenizer-level representation of a piece of text.

Example:

```text
"learning"
```

---

### Token ID

A token ID is the integer identifier assigned to that token.

Example:

```text
5678
```

---

### Embedding

An embedding is the initial learned numerical vector associated with the token ID.

Example:

```text
[0.1, -0.3, 0.8, ...]
```

---

### Hidden State

A hidden state is a context-dependent representation produced after the token representation has passed through Transformer computation.

Conceptually:

```text
Embedding
   ↓
Transformer
   ↓
Hidden State
```

This means:

```text
Token
   ↓
Token ID
   ↓
Embedding
   ↓
Transformer Processing
   ↓
Contextual Hidden State
```

The embedding and hidden state are therefore different objects.

The embedding is an initial learned representation, while the hidden state contains contextual information produced by the Transformer.

---

## 13 — Position Matters

Knowing which tokens exist is not enough.

The model must also understand where tokens occur in the sequence.

Consider:

```text
"dog bites man"
```

and:

```text
"man bites dog"
```

They contain the same basic words, but their meanings are different.

Therefore, the model needs a way to represent sequence position.

The general principle is:

```text
Token Identity + Position
           ↓
Contextual Processing
```

The exact method used to represent positional information depends on the model architecture.

Different model families may use different positional mechanisms.

For Day 1, the important idea is simply:

> **The model needs information about both token identity and token position.**

We do not need to memorize a specific positional-encoding implementation yet.

---

## 14 — Decoder-Only Transformer

The Local AI Assistant will use a pretrained language model built around a decoder-style Transformer architecture.

At a high level:

```text
Token IDs
   ↓
Token Embeddings
   +
Position Information
   ↓
Transformer Blocks
   ↓
Final Hidden States
   ↓
Language Modeling Head
   ↓
Logits
```

A Transformer block contains several important mechanisms.

At a high level, we should recognize:

```text
Self-Attention
Feed-Forward Network
Normalization
Residual Connections
```

These components work together to transform the input token representations into contextual hidden states.

The exact implementation differs between model families.

For Day 1, the objective is to understand the architecture at a conceptual level rather than study every internal implementation detail.

---

## 14.1 — Self-Attention

The core intuition behind self-attention is that a token's representation can incorporate information from other relevant tokens in its context.

Consider:

```text
"The animal didn't cross the road because it was tired."
```

To interpret:

```text
"it"
```

the model needs information from the surrounding context.

Self-attention is one of the main mechanisms that allows contextual relationships between tokens to be modeled.

Conceptually:

```text
Token
  ↓
Looks at relevant context
  ↓
Updated contextual representation
```

For decoder-only autoregressive models, this attention is causal, meaning the model cannot use future tokens that have not yet been generated.

---

## 14.2 — Q / K / V Preview

We will study the mathematical and implementation details later.

For now, the mental model is:

```text
Hidden States
      ↓
   Q   K   V
      ↓
 Attention
      ↓
Contextual Representation
```

The general attention equation is:

\[
Attention(Q,K,V)
=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

For Day 1, understanding what Q, K, and V represent at a high level is enough.

The deeper self-attention mechanics will be studied in Day 2.

---

## 14.3 — Causal Attention

During autoregressive language modeling, the model must predict the next token using only information that is available so far.

Conceptually:

```text
Token 1
  ↓
can see Token 1
```

```text
Token 2
  ↓
can see Token 1 + Token 2
```

```text
Token 3
  ↓
can see Token 1 + Token 2 + Token 3
```

But:

```text
Token 2
   ✗
cannot see Token 3
```

This causal constraint prevents the model from looking ahead during next-token prediction.

Therefore:

```text
Current Context
      ↓
Predict Next Token
```

rather than:

```text
Current Context + Future Tokens
      ↓
Predict Next Token
```

Causal attention is one of the foundations of decoder-only autoregressive generation.

---

## 14.4 — Next-Token Prediction

Suppose the input is:

```text
"I am learning"
```

The model does not conceptually generate an entire paragraph in one single prediction step.

Instead, it produces scores for possible next tokens.

Conceptually:

```text
"AI"          → 0.42
"machine"     → 0.21
"Python"      → 0.08
"about"       → 0.05
...
```

These values are illustrative.

The exact values depend on the model and context.

The important idea is:

```text
Current Tokens
      ↓
Predict Next Token
      ↓
Append Next Token
      ↓
Predict Again
```

For example:

```text
"I am learning"
        ↓
     predict
        ↓
       "AI"
```

Then:

```text
"I am learning AI"
        ↓
     predict
        ↓
   "engineering"
```

And again:

```text
"I am learning AI engineering"
        ↓
     predict
        ↓
       "..."
```

This repeated process is called **autoregressive generation**.

---

## 14.5 — Logits

Before converting the model's output scores into probabilities, the model produces **logits**.

The simplified pipeline is:

```text
Transformer
    ↓
Hidden States
    ↓
Language Modeling Head
    ↓
Logits
```

If the vocabulary contains many thousands of possible tokens, the model produces a corresponding score for each possible next token.

Conceptually:

```text
Vocabulary
────────────────
token 1   → score
token 2   → score
token 3   → score
...
token N   → score
```

These raw scores are the logits.

A probability distribution can then be derived from the logits.

Sampling strategies such as:

```text
Temperature
Top-k
Top-p
```

will be studied later in W15.

---

## 15 — Context Window

An LLM operates within a limited **context window**.

The context window represents the amount of tokenized context that the model can process as part of a single sequence.

Conceptually:

```text
┌─────────────────────────────┐
│        Context Window       │
│                             │
│ System Instruction          │
│ User Message                │
│ Conversation History        │
│ Current Prompt              │
│ Generated Tokens            │
│                             │
└─────────────────────────────┘
```

The context window is measured in **tokens**, not characters.

This matters because token count affects how much information can fit into the model's available context.

Token count becomes important for:

- Chat history
- RAG
- Long documents
- Memory
- Latency
- Memory usage

The relationship can be simplified as:

```text
Text Length
    ↓
Token Count
    ↓
Available Context
    ↓
Computation / Memory Implications
```

Two prompts may look similar as strings while having very different token counts.

For example:

```text
Prompt A → 100 tokens
Prompt B → 900 tokens
```

From an inference-engineering perspective, these are not equivalent workloads.

The longer sequence generally requires more input processing.

This is why token counting is a critical part of LLM engineering.

The concept of the context window will become especially important later when we work with:

```text
W16 → RAG
W17 → Advanced RAG
W18 → Agents
W19 → Inference Engineering
```

The key mental model is:

```text
Context Window
=
The token budget available to the model for processing the current sequence.
```

---

# Mental Model So Far

At this point, the complete conceptual pipeline is:

```text
Human Text
   ↓
Tokenizer
   ↓
Tokens
   ↓
Token IDs
   ↓
Embedding Lookup
   ↓
Embedding Vectors
   +
Position Information
   ↓
Decoder-Only Transformer
   ↓
Contextual Hidden States
   ↓
Language Modeling Head
   ↓
Logits
```

And later, the generation process becomes:

```text
Logits
   ↓
Probability Distribution
   ↓
Decoding / Sampling
   ↓
Next Token
   ↓
Append Token
   ↓
Generate Again
```

The central idea is:

> **An LLM does not process raw human language directly. Text is first tokenized, mapped to token IDs, transformed into numerical representations, processed through a Transformer, and converted into scores for possible next tokens.**

# 1. Tokenization, Token IDs, and Embeddings

This notebook explores how raw text is transformed into tokens,
token IDs, and model-ready numerical representations.

`Experiments`

- Tokenization
- Token IDs
- Special Tokens
- Token Counting
- English Text
- Arabic Text
- Mixed Arabic + English
- Code
- Numbers and Punctuation
- Padding and Truncation
- Attention Masks
- Chat Templates
- Embedding Inspection
- Tensor Shape Tracking
- Mini Forward Pass

## 1.1 Python Environment Check

In [1]:
# Validate the local Python environment

import sys

print("Python:", sys.version)

Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]


### Result Interpretation

The notebook is running on Python 3.13.0.

This confirms that the selected Python kernel is active and available for the Week 15 Day 1 experiments.

### Finding

The project is running on the intended Python environment.

## 1.2 Transformers Installation Check

In [3]:
# Install Hugging Face Transformers

%pip install -U transformers

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 1.3 MB/s eta 0:00:10
   - -------------------------------------- 0.5/12.6 MB 1.3 MB/s eta 0:00:10
   --- ------------------------------------ 1.0/12.6 MB 1.4 MB/s eta 0:00:09
   ---- ----------------------------------- 1.3/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.6/12.6 MB 1.4 MB/s eta 0:00:08
   ------ --------------------------------- 2.1/12.6 MB 1.4 MB/s eta 0:00:08
   ------- -------------------------------- 2.4/12.6 MB 1.4 MB/s eta 0:00:08
   -------- ------------------------------- 2.6/12.6 MB 1.5 MB/s eta 0:00:07
   --------- ------------------------------ 2.9/12.6 MB 1.4 MB/s eta 0:00:07
   ---------- ----------------------------- 3.1/12.6 MB 1.4 MB/s eta 0:00:07
   ----------- ---------------------------- 3.7/12.6 MB 1.4 MB/s eta 0:00:07
   ----------


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Verify the Transformers installation

import transformers

print("Transformers:", transformers.__version__)

d:\Projects\Local AI Assistan\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Transformers: 5.18.0


### Result Interpretation

Transformers 5.18.0 is installed and can be imported successfully.

This confirms that the Hugging Face Transformers library is available in the project environment.

### Finding

The Transformers dependency is correctly installed and ready for LLM tokenizer experiments.

## 1.3 Qwen3.8-27B Tokenizer Loading

In [12]:
# Load the Qwen3.8-27B tokenizer

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.8-27B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)

d:\Projects\Local AI Assistan\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahmu\.cache\huggingface\hub\models--Qwen--Qwen3.8-27B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer: Qwen3_5Tokenizer
Vocabulary size: 248044


### Result Interpretation

The tokenizer for the Qwen3.8-27B model was loaded successfully.

The tokenizer reports a vocabulary size of 248,044 tokens.

### Finding

The project is using the tokenizer associated with the selected Qwen3.8-27B model setup.

## 1.4 Basic English Tokenization

In [ ]:
# Tokenize a simple English sentence

text = "I love machine learning."

tokens = tokenizer.tokenize(text)

print("Text:", text)
print("Tokens:", tokens)
print("Token count:", len(tokens))

Text: I love machine learning.
Tokens: ['I', 'Ġlove', 'Ġmachine', 'Ġlearning', '.']
Token count: 5


### Result Interpretation

The sentence:

```text
I love machine learning.
```
was tokenized into five token pieces:
```
I
Ġlove
Ġmachine
Ġlearning
.
```
The Ġ marker represents a tokenizer-specific representation associated with a preceding whitespace boundary.

### Finding

A token is not necessarily equivalent to a word or a character. Token boundaries depend on the tokenizer.

## 1.5 Token IDs 


In [5]:
# Convert tokens into token IDs

encoded = tokenizer(text)

input_ids = encoded["input_ids"]

print("Tokens:", tokens)
print("Token IDs:", input_ids)
print("Token count:", len(input_ids))

Tokens: ['I', 'Ġlove', 'Ġmachine', 'Ġlearning', '.']
Token IDs: [40, 2854, 5484, 6618, 13]
Token count: 5


### Result Interpretation

The five tokens were converted into integer token IDs:

```text
I          → 40
Ġlove      → 2854
Ġmachine   → 5484
Ġlearning  → 6618
.          → 13
```
These integers are vocabulary identifiers used by the model input pipeline.

The numerical value of a token ID does not represent semantic meaning. It functions as an index for retrieving the corresponding token representation.

### Finding

The tokenizer converts discrete token pieces into integer IDs that can be processed by the model.

## 1.6 Token-to-ID Mapping


In [6]:
# Inspect the token-to-ID mapping

for token, token_id in zip(tokens, input_ids):
    print(f"{token!r:15} → {token_id}")

'I'             → 40
'Ġlove'         → 2854
'Ġmachine'      → 5484
'Ġlearning'     → 6618
'.'             → 13


### Result Interpretation

The explicit token-to-ID mapping confirms the relationship between each token and its vocabulary identifier.

The mapping is deterministic for the selected tokenizer and input text.

### Finding

Each token in the encoded sequence corresponds to one integer identifier in the tokenizer vocabulary.

## 1.7 Decode Token IDs Back to Text

In [7]:
# Decode token IDs back into text

decoded_text = tokenizer.decode(
    input_ids,
    skip_special_tokens=False
)

print("Original:", text)
print("Decoded :", decoded_text)

Original: I love machine learning.
Decoded : I love machine learning.


### Result Interpretation

The encoded token IDs were decoded back into text successfully.

```text
Original: I love machine learning.
Decoded : I love machine learning.
```
The decoded text matches the original input for this example.

### Finding

The tokenizer successfully performs the round-trip transformation:
```
Text
 ↓
Token IDs
 ↓
Decoded Text

## 1.8 Character, Word, and Token Counts


In [8]:
# Compare character, word, and token counts

character_count = len(text)
word_count = len(text.split())
token_count = len(input_ids)

print("Text:", text)
print("Character count:", character_count)
print("Word count:", word_count)
print("Token count:", token_count)

Text: I love machine learning.
Character count: 24
Word count: 4
Token count: 5


### Result Interpretation

The input text contains:

```text
Characters: 24
Words: 4
Tokens: 5
```
These values are different because tokenization does not operate directly on natural language words or individual characters.

### Finding

Character count, word count, and token count are distinct measurements. LLM context is measured in tokens rather than words or characters.

## 1.9 Arabic Tokenization


In [9]:
# Tokenize an Arabic sentence

arabic_text = "أنا أحب تعلم الذكاء الاصطناعي."

arabic_tokens = tokenizer.tokenize(arabic_text)
arabic_ids = tokenizer(arabic_text)["input_ids"]

print("Text:", arabic_text)
print("Tokens:", arabic_tokens)
print("Token IDs:", arabic_ids)
print("Character count:", len(arabic_text))
print("Word count:", len(arabic_text.split()))
print("Token count:", len(arabic_ids))

Text: أنا أحب تعلم الذكاء الاصطناعي.
Tokens: ['Ø£', 'ÙĨØ§', 'ĠØ£ØŃØ¨', 'ĠØªØ¹ÙĦÙħ', 'ĠØ§ÙĦØ°', 'ÙĥØ§Ø¡', 'ĠØ§ÙĦØ§Øµ', 'Ø·ÙĨ', 'Ø§Ø¹ÙĬ', '.']
Token IDs: [67321, 148415, 226577, 198868, 149634, 218326, 196720, 149158, 198240, 13]
Character count: 30
Word count: 5
Token count: 10


### Result Interpretation

The Arabic sentence contains:

```text
Characters: 30
Words: 5
Tokens: 10
```
Several words are represented by multiple token pieces.

This demonstrates that tokenization behavior depends on the language and the tokenizer vocabulary.

### Finding

A small number of words can correspond to a larger number of tokens. Word count alone is not sufficient for estimating an LLM input size.


## 1.10 Arabic Token Inspection


In [10]:
# Inspect Arabic tokens using repr()

for i, (token, token_id) in enumerate(zip(arabic_tokens, arabic_ids)):
    print(f"{i:02d} | {token!r} → {token_id}")

00 | 'Ø£' → 67321
01 | 'ÙĨØ§' → 148415
02 | 'ĠØ£ØŃØ¨' → 226577
03 | 'ĠØªØ¹ÙĦÙħ' → 198868
04 | 'ĠØ§ÙĦØ°' → 149634
05 | 'ÙĥØ§Ø¡' → 218326
06 | 'ĠØ§ÙĦØ§Øµ' → 196720
07 | 'Ø·ÙĨ' → 149158
08 | 'Ø§Ø¹ÙĬ' → 198240
09 | '.' → 13


### Result Interpretation

Inspecting each token together with its token ID shows how the tokenizer decomposes the Arabic input into smaller pieces.

Some words are split across multiple tokens rather than represented by a single token.

The displayed token strings may use tokenizer-specific byte-level representations, while the token IDs remain the numerical identifiers used by the model pipeline.

### Finding

Token boundaries can occur inside words, and token IDs provide the stable numerical representation used by the tokenizer and model.

## 1.11 Arabic Token ID Decoding

In [11]:
# Decode each Arabic token ID individually

for i, token_id in enumerate(arabic_ids):
    decoded_token = tokenizer.decode(
        [token_id],
        skip_special_tokens=False
    )
    
    print(f"{i:02d} | ID: {token_id:<7} | Decoded: {decoded_token!r}")

00 | ID: 67321   | Decoded: 'أ'
01 | ID: 148415  | Decoded: 'نا'
02 | ID: 226577  | Decoded: ' أحب'
03 | ID: 198868  | Decoded: ' تعلم'
04 | ID: 149634  | Decoded: ' الذ'
05 | ID: 218326  | Decoded: 'كاء'
06 | ID: 196720  | Decoded: ' الاص'
07 | ID: 149158  | Decoded: 'طن'
08 | ID: 198240  | Decoded: 'اعي'
09 | ID: 13      | Decoded: '.'


### Result Interpretation

Each Arabic token ID was decoded individually to reveal the text fragment represented by that ID.

The decoded fragments reconstruct the original sentence when combined in sequence.

### Finding

A complete word does not necessarily correspond to a single token. Multiple token pieces can be combined to represent one word.

## 1.12 Mixed Langue

In [13]:
# Tokenize mixed Arabic and English text

mixed_text = "أتعلم Machine Learning باستخدام Python."

mixed_tokens = tokenizer.tokenize(mixed_text)
mixed_ids = tokenizer(mixed_text)["input_ids"]

print("Text:", mixed_text)
print("Tokens:", mixed_tokens)
print("Token IDs:", mixed_ids)
print("Character count:", len(mixed_text))
print("Word count:", len(mixed_text.split()))
print("Token count:", len(mixed_ids))

Text: أتعلم Machine Learning باستخدام Python.
Tokens: ['Ø£Øª', 'Ø¹ÙĦÙħ', 'ĠMachine', 'ĠLearning', 'ĠØ¨Ø§Ø³ØªØ®Ø¯Ø§Ùħ', 'ĠPython', '.']
Token IDs: [160409, 171519, 12588, 20271, 179677, 12654, 13]
Character count: 39
Word count: 5
Token count: 7


### Result Interpretation

The mixed-language input:

```text
أتعلم Machine Learning باستخدام Python.
```

contains:

```text
Character count: 39
Word count: 5
Token count: 7
```

The tokenizer produced a single sequence containing Arabic text, English technical terms, and punctuation.

The observed tokenization shows that different parts of the same sentence can have different tokenization patterns. Some words are represented by a single token, while other words may be split into multiple tokens.

The `Ġ` marker appearing before several English tokens represents a tokenizer-specific whitespace boundary.

The displayed Arabic token strings use a byte-level representation in the notebook output, so the decoded text and token IDs provide a clearer representation for analysis.

### Finding

Mixed-language text is tokenized according to the tokenizer vocabulary rather than natural word boundaries.

The same tokenizer can process Arabic and English within a single sequence, but tokenization efficiency can vary across languages and individual words.

The number of words is therefore not sufficient to estimate the number of tokens processed by the model. Token count is the relevant measurement for understanding LLM context usage.

## 1.13 Code Tokenization

In [14]:
# Tokenize a code snippet

code = """def add_numbers(a, b):
    return a + b
"""

code_tokens = tokenizer.tokenize(code)
code_ids = tokenizer(code)["input_ids"]

print("Code:")
print(code)

print("Tokens:", code_tokens)
print("Token IDs:", code_ids)

print("Character count:", len(code))
print("Word count:", len(code.split()))
print("Token count:", len(code_ids))

Code:
def add_numbers(a, b):
    return a + b

Tokens: ['def', 'Ġadd', '_numbers', '(a', ',', 'Ġb', '):', 'Ċ', 'ĠĠĠ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb', 'Ċ']
Token IDs: [727, 884, 31871, 2784, 11, 292, 1590, 198, 262, 460, 264, 478, 292, 198]
Character count: 40
Word count: 7
Token count: 14


### Result Interpretation

The code snippet contains 40 characters, 7 whitespace-separated words, and 14 tokens.

The tokenizer does not treat programming code as a sequence of complete words. It also represents identifiers, punctuation, operators, whitespace, indentation, and line breaks as part of the token sequence.

For example, the identifier `add_numbers` is split into multiple tokens, while punctuation such as `,` can appear as an individual token.

The `Ċ` marker represents a line break, while the `Ġ` marker indicates a whitespace-related boundary in the tokenizer representation.

The token IDs provide the numerical representation corresponding to these tokenizer outputs.

### Finding

Code tokenization follows the tokenizer vocabulary and segmentation rules rather than programming-language word boundaries.

A short code snippet can therefore produce significantly more tokens than its whitespace-separated word count suggests.

This demonstrates that token count is a more relevant measurement than word count when estimating how much code or text an LLM processes.

## 1.14 Numbers & Punctuation

In [15]:
# Tokenize numbers and punctuation

text = "The price is 12,500.75 USD! Is it 100% correct?"

tokens = tokenizer.tokenize(text)
input_ids = tokenizer(text)["input_ids"]

print("Text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(input_ids)

print("\nCharacter count:", len(text))
print("Word count:", len(text.split()))
print("Token count:", len(input_ids))

Text:
The price is 12,500.75 USD! Is it 100% correct?

Tokens:
['The', 'Ġprice', 'Ġis', 'Ġ', '1', '2', ',', '5', '0', '0', '.', '7', '5', 'ĠUSD', '!', 'ĠIs', 'Ġit', 'Ġ', '1', '0', '0', '%', 'Ġcorrect', '?']

Token IDs:
[760, 3240, 369, 220, 16, 17, 11, 20, 15, 15, 13, 22, 20, 18954, 0, 2091, 424, 220, 16, 15, 15, 4, 4252, 30]

Character count: 47
Word count: 9
Token count: 24


### Result Interpretation

The input contains 47 characters, 9 whitespace-separated words, and 24 tokens.

The number `12,500.75` is not represented as a single token. It is segmented into multiple tokens containing individual digits and punctuation.

Similarly, `100%` is segmented into separate digit tokens followed by a percentage symbol.

In contrast, several ordinary words are represented by single tokens. The tokenizer therefore applies segmentation rules based on its vocabulary rather than human-defined word or number boundaries.

The `Ġ` marker represents a tokenizer-specific whitespace boundary, while punctuation such as commas, periods, exclamation marks, question marks, and percentage symbols can appear as separate tokens.

### Finding

Numbers and punctuation can produce substantially different tokenization patterns from ordinary words.

A numerical value that appears as one logical unit to a human can require multiple tokens for the tokenizer.

Therefore, word count and visible text structure are not reliable measures of the number of tokens processed by an LLM.